### Installation

In [2]:
print("STARTING")

STARTING


In [3]:
%%capture
import os, importlib.util
!pip install --upgrade -qqq uv
if importlib.util.find_spec("torch") is None or "COLAB_" in "".join(os.environ.keys()):
    try: import numpy, PIL; get_numpy = f"numpy=={numpy.__version__}"; get_pil = f"pillow=={PIL.__version__}"
    except: get_numpy = "numpy"; get_pil = "pillow"
    !uv pip install -qqq \
        "torch>=2.8.0" "triton>=3.4.0" {get_numpy} {get_pil} torchvision bitsandbytes "transformers==4.56.2" \
        "unsloth_zoo[base] @ git+https://github.com/unslothai/unsloth-zoo" \
        "unsloth[base] @ git+https://github.com/unslothai/unsloth" \
        git+https://github.com/triton-lang/triton.git@0add68262ab0a2e33b84524346cb27cbb2787356#subdirectory=python/triton_kernels
elif importlib.util.find_spec("unsloth") is None:
    !uv pip install -qqq unsloth
!uv pip install --upgrade --no-deps transformers==4.56.2 tokenizers trl==0.22.2 unsloth unsloth_zoo

### Unsloth

We're about to demonstrate the power of the new OpenAI GPT-OSS 20B model through an inference example. For our `bnb-4bit` version, use this [notebook](https://colab.research.google.com/github/unslothai/notebooks/blob/main/nb/GPT_OSS_BNB_(20B)-Inference.ipynb) instead.

**We're using OpenAI's MXFP4 Triton kernels combined with Unsloth's kernels!**

In [4]:
from unsloth import FastLanguageModel
import torch

# 4bit pre quantized models we support for 4x faster downloading + no OOMs.
fourbit_models = [
    "unsloth/gpt-oss-20b-unsloth-bnb-4bit", # 20B model using bitsandbytes 4bit quantization
    "unsloth/gpt-oss-120b-unsloth-bnb-4bit",
    "unsloth/gpt-oss-20b", # 20B model using MXFP4 format
    "unsloth/gpt-oss-120b",
] # More models at https://huggingface.co/unsloth

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = "unsloth/gpt-oss-20b",
    dtype = None, # None for auto detection
    max_seq_length = 4096, # Choose any for long context!
    load_in_4bit = False,  # 4 bit quantization to reduce memory
    full_finetuning = False, # [NEW!] We have full finetuning now!
    # token = "hf_...", # use one if using gated models
)

🦥 Unsloth: Will patch your computer to enable 2x faster free finetuning.


/usr/local/lib/python3.12/dist-packages/torchao/quantization/quant_api.py:2525: SyntaxWarning: invalid escape sequence '\.'
  * regex for parameter names, must start with `re:`, e.g. `re:language\.layers\..+\.q_proj.weight`.


🦥 Unsloth Zoo will now patch everything to make training faster!
==((====))==  Unsloth 2025.12.9: Fast Gpt_Oss patching. Transformers: 4.56.2.
   \\   /|    NVIDIA L4. Num GPUs = 1. Max memory: 22.161 GB. Platform: Linux.
O^O/ \_/ \    Torch: 2.9.0+cu126. CUDA: 8.9. CUDA Toolkit: 12.6. Triton: 3.5.0
\        /    Bfloat16 = TRUE. FA [Xformers = None. FA2 = False]
 "-____-"     Free license: http://github.com/unslothai/unsloth
Unsloth: Fast downloading is enabled - ignore downloading bars which are red colored!
Unsloth: QLoRA and full finetuning all not selected. Switching to 16bit LoRA.


model.safetensors.index.json: 0.00B [00:00, ?B/s]

model-00000-of-00002.safetensors:   0%|          | 0.00/4.79G [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/165 [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenizer.json:   0%|          | 0.00/27.9M [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/446 [00:00<?, ?B/s]

chat_template.jinja: 0.00B [00:00, ?B/s]

In [ ]:
LOG_FILE = "D:\\ALL CODES\\AIMO Progress Prize 3\\LOG\\LOG.txt"

In [ ]:
# --- Python execution tool + pass-1 TIR loop (ported from AIMO-4 style) ---
# This implements a *stateful* python executor and a simple Tool-Integrated Reasoning loop
# where the model can emit code multiple times in one solve pass.

import re
import io
import math
import types
import traceback
import contextlib
from dataclasses import dataclass
from typing import Any, Dict, Optional, List
from datetime import datetime
from pathlib import Path

import torch
from unsloth import FastLanguageModel

# Ensure Unsloth inference optimizations are active.
FastLanguageModel.for_inference(model)

# Always log to files in the notebook's working directory.
DEFAULT_LOG_FILENAME = "python_tool_log2.txt"
DEFAULT_MODEL_LOG_FILENAME = "model_thinking_log.txt"


def _maybe_import_sympy() -> Optional[types.ModuleType]:
    try:
        import sympy as sp  # type: ignore
        return sp
    except Exception:
        return None


def _add_common_libs(code: str) -> str:
    """Prepend common math libraries."""
    header = ["import math", "import numpy as np"]
    sp = _maybe_import_sympy()
    if sp is not None:
        header.extend(["import sympy as sp", "from sympy import *"])
    return "\n".join(header) + "\n" + code.lstrip()


def _ensure_last_expr_printed(code: str) -> str:
    """If the final line looks like an expression, wrap it in print(...)."""
    lines = [ln.rstrip() for ln in code.strip().splitlines()]
    if not lines:
        return code
    last = lines[-1].strip()
    if not last:
        return "\n".join(lines)
    if last.startswith(("print(", "import ", "from ", "#", "for ", "while ", "if ", "def ", "class ", "return ", "assert ")):
        return "\n".join(lines)
    lines[-1] = f"print({last})"
    return "\n".join(lines)


@dataclass
class PythonExecResult:
    ok: bool
    output: str
    error: str = ""


class LocalPythonTool:
    """Stateful python execution tool (keeps globals across calls)."""
    def __init__(self):
        self._globals: Dict[str, Any] = {"__name__": "__main__"}
        try:
            import numpy as np  # type: ignore
            self._globals["np"] = np
        except Exception:
            pass
        sp = _maybe_import_sympy()
        if sp is not None:
            self._globals["sp"] = sp
        self._globals["math"] = math

    def run(self, code: str) -> PythonExecResult:
        code = _add_common_libs(code)
        code = _ensure_last_expr_printed(code)
        stdout = io.StringIO()
        try:
            with contextlib.redirect_stdout(stdout):
                exec(code, self._globals, self._globals)
            out = stdout.getvalue().strip()
            return PythonExecResult(ok=True, output=out)
        except Exception:
            err = traceback.format_exc()
            out = stdout.getvalue().strip()
            return PythonExecResult(ok=False, output=out, error=err)


def _get_log_file_path() -> Path:
    """Python tool log file path in the notebook's current working directory."""
    return Path.cwd() / DEFAULT_LOG_FILENAME


def _get_model_log_file_path() -> Path:
    """Model transcript/thinking log file path in the notebook's current working directory."""
    return Path.cwd() / DEFAULT_MODEL_LOG_FILENAME


def _reset_log_file(path: Path, header: str = "# Log") -> None:
    """Delete and recreate a log file (start fresh each solve call)."""
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        if path.exists():
            path.unlink()
    except Exception:
        pass
    try:
        path.write_text(header.rstrip() + "\n", encoding="utf-8")
    except Exception:
        pass


def _append_tool_log(*, code: str, ok: bool, stdout: str, error: str) -> None:
    """Append an execution log entry to the python tool log file."""
    try:
        path = _get_log_file_path()
        if not path.exists():
            # If something called logging without reset, still create a minimal file.
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text("# Python tool log\n", encoding="utf-8")

        ts = datetime.now().isoformat(timespec="seconds")
        lines = [
            f"\n--- PYTHON TOOL EXEC {ts} ---",
            f"status: {'OK' if ok else 'FAIL'}",
            "code:",
            (code or "").rstrip(),
            "stdout:",
            (stdout if stdout else "(no stdout)"),
        ]
        if not ok:
            lines.extend(["error:", (error or "").rstrip()])
        lines.append("--- END ---\n")
        with path.open("a", encoding="utf-8") as f:
            f.write("\n".join(lines))
    except Exception:
        # Never break solving due to logging failures.
        pass


def _append_model_log(*, role: str, content: str) -> None:
    """Append model/user/tool transcript to the model log file."""
    try:
        path = _get_model_log_file_path()
        if not path.exists():
            path.parent.mkdir(parents=True, exist_ok=True)
            path.write_text("# Model transcript log\n", encoding="utf-8")

        ts = datetime.now().isoformat(timespec="seconds")
        safe_role = (role or "").strip() or "unknown"
        text = (content or "").rstrip()
        block = [
            f"\n--- {safe_role.upper()} {ts} ---",
            text if text else "(empty)",
            "--- END ---\n",
        ]
        with path.open("a", encoding="utf-8") as f:
            f.write("\n".join(block))
    except Exception:
        # Never break solving due to logging failures.
        pass


def _tail_lines(text: str, n: int = 30) -> str:
    lines = (text or "").splitlines()
    if len(lines) <= n:
        return text or ""
    return "\n".join(lines[-n:])


_PY_FENCE_RE = re.compile(r"```(?:python|py)\s*(.*?)```", re.DOTALL | re.IGNORECASE)
_META_MARKERS = ("commentary", "assistant", "analysis", "final", "FINAL:", "```")


def _truncate_at_meta(text: str) -> str:
    """Best-effort: strip any injected chat/meta text accidentally glued onto code."""
    if not text:
        return text
    earliest: Optional[int] = None
    for m in _META_MARKERS:
        i = text.find(m)
        if i != -1:
            earliest = i if earliest is None else min(earliest, i)
    if earliest is not None:
        text = text[:earliest]
    return text.strip()


def _sanitize_extracted_code(code: str) -> str:
    """Trim any accidental chat/meta text that got captured with the code."""
    if not code:
        return code
    # First cut off any glued-on meta markers (works even if they appear mid-line).
    code = _truncate_at_meta(code)
    if not code:
        return code
    # Then, also stop at whole-line meta markers if present.
    stop_prefixes = ("commentary", "assistant", "analysis", "final", "FINAL:", "```")
    cleaned: List[str] = []
    for line in code.splitlines():
        stripped = line.lstrip()
        if stripped.startswith(stop_prefixes):
            break
        cleaned.append(line)
    return "\n".join(cleaned).strip()


def _extract_last_to_python_code(text: str) -> Optional[str]:
    """Extract code after the last 'to=python ... code' marker (robust to inline glue)."""
    if not text:
        return None
    low = text.lower()
    idx = low.rfind("to=python")
    if idx == -1:
        return None
    # Find the nearest 'code' after that marker.
    code_idx = low.find("code", idx)
    if code_idx == -1:
        return None
    start = code_idx + len("code")
    # Skip separators/spaces/newlines after 'code'.
    while start < len(text) and text[start] in " :\t\r\n":
        start += 1
    snippet = text[start:]
    snippet = _sanitize_extracted_code(snippet)
    return snippet or None


def _extract_last_python_snippet(text: str) -> Optional[str]:
    """Extract the last python snippet from either fenced blocks or `to=python` style."""
    text = text or ""
    fenced = _PY_FENCE_RE.findall(text)
    if fenced:
        snippet = _sanitize_extracted_code(fenced[-1].strip())
        return snippet or None
    to_py = _extract_last_to_python_code(text)
    return to_py or None


_BOXED_RE = re.compile(r"\\boxed\{([^}]*)\}")
_FINAL_BOXED_RE = re.compile(r"FINAL\s*:?\s*\\boxed\{([^}]*)\}", re.IGNORECASE)


def _extract_final_only(text: str) -> Optional[str]:
    """Return only `FINAL: \\boxed{...}` if present, else None."""
    text = (text or "").strip()
    m = list(_FINAL_BOXED_RE.finditer(text))
    if m:
        val = m[-1].group(1).strip()
        return f"FINAL: \\boxed{{{val}}}"
    m2 = list(_BOXED_RE.finditer(text))
    if m2:
        val = m2[-1].group(1).strip()
        return f"FINAL: \\boxed{{{val}}}"
    return None


def _generate_assistant(
    messages: List[Dict[str, str]],
    reasoning_effort: str = "high",
    max_new_tokens: int = 1024,
    temperature: float = 0.0,
    top_p: float = 1.0,
 ) -> str:
    """Generate assistant text and return only the newly generated part."""
    inputs = tokenizer.apply_chat_template(
        messages,
        add_generation_prompt=True,
        return_tensors="pt",
        return_dict=True,
        reasoning_effort=reasoning_effort,
    ).to("cuda")
    prompt_len = inputs["input_ids"].shape[-1]
    with torch.inference_mode():
        out = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            do_sample=temperature > 0,
            temperature=max(temperature, 1e-6),
            top_p=top_p,
        )
    gen_tokens = out[0][prompt_len:]
    return tokenizer.decode(gen_tokens, skip_special_tokens=True).strip()


def solve_with_tir_pass1(
    question: str,
    *,
    reasoning_effort: str = "high",
    max_tool_calls: int = 3,
    max_new_tokens: int = 1024,
    temperature: float = 0.0,
    verbose_tool: bool = True,
 ) -> str:
    """Pass-1 Tool-Integrated Reasoning: model may call python up to `max_tool_calls` times."""
    tool = LocalPythonTool()
    system = (
        "You are solving a math problem. You may use a Python tool for computation/verification.\n\n"
        "Tool-use rule: When you want to run code, output ONLY a python fenced code block like:\n"
        "```python\n<code>\n```\n\n"
        "Do NOT include words like 'assistant', 'analysis', 'commentary', or 'to=python' around the code.\n"
        "After I reply with the Python output, continue solving.\n"
        "When finished, output the final answer as: FINAL: \\boxed{...}."
    )
    messages: List[Dict[str, str]] = [
        {"role": "system", "content": system},
        {"role": "user", "content": question},
    ]

    # Reset log files for each run.
    tool_log_path = _get_log_file_path()
    model_log_path = _get_model_log_file_path()
    _reset_log_file(tool_log_path, header="# Python tool log")
    _reset_log_file(model_log_path, header="# Model transcript log")
    _append_model_log(role="system", content=system)
    _append_model_log(role="user", content=question)
    if verbose_tool:
        print(f"[python] tool log file: {tool_log_path}")
        print(f"[model] transcript log file: {model_log_path}")

    for _ in range(max_tool_calls + 1):
        assistant = _generate_assistant(
            messages,
            reasoning_effort=reasoning_effort,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
        )
        messages.append({"role": "assistant", "content": assistant})
        _append_model_log(role="assistant", content=assistant)

        final_only = _extract_final_only(assistant)
        if final_only is not None:
            _append_model_log(role="final", content=final_only)
            return final_only

        code = _extract_last_python_snippet(assistant)
        if not code:
            # No tool call and no FINAL found: ask once for FINAL-only.
            prompt = "Reply with FINAL only: FINAL: \\boxed{...}"
            messages.append({
                "role": "user",
                "content": prompt,
            })
            _append_model_log(role="user", content=prompt)
            assistant2 = _generate_assistant(
                messages,
                reasoning_effort=reasoning_effort,
                max_new_tokens=max_new_tokens,
                temperature=temperature,
            )
            _append_model_log(role="assistant", content=assistant2)
            final2 = _extract_final_only(assistant2)
            if final2 is not None:
                _append_model_log(role="final", content=final2)
                return final2
            return assistant2.strip()

        # Print the exact code being executed (for debugging).
        if verbose_tool:
            print("[python] code (about to run):\n" + code)

        result = tool.run(code)
        _append_tool_log(code=code, ok=result.ok, stdout=result.output, error=result.error)

        if verbose_tool:
            print("[python] execution: SUCCESS" if result.ok else "[python] execution: FAILED")
            if result.output:
                print("[python] stdout:\n" + result.output)
            if not result.ok and result.error:
                print("[python] error (tail):\n" + _tail_lines(result.error, 30))

        if result.ok:
            tool_msg = "[PYTHON_OK]\n" + (result.output if result.output else "(no stdout)")
        else:
            tool_msg = (
                "[PYTHON_ERROR]\n"
                + (result.output + "\n" if result.output else "")
                + result.error
            )
        user_continue = "Python output:\n" + tool_msg + "\n\nContinue. Remember: when done output FINAL only."
        messages.append({
            "role": "user",
            "content": user_continue,
        })
        _append_model_log(role="tool", content=f"python\n{code}\n\n{tool_msg}")
        _append_model_log(role="user", content=user_continue)

    final_prompt = "No more python tool calls allowed. Output FINAL only: FINAL: \\boxed{...}"
    messages.append({
        "role": "user",
        "content": final_prompt,
    })
    _append_model_log(role="user", content=final_prompt)
    assistant3 = _generate_assistant(
        messages,
        reasoning_effort=reasoning_effort,
        max_new_tokens=max_new_tokens,
        temperature=temperature,
    )
    _append_model_log(role="assistant", content=assistant3)
    final3 = _extract_final_only(assistant3)
    if final3 is not None:
        _append_model_log(role="final", content=final3)
        return final3
    return assistant3.strip()

In [44]:
# Example: pass-1 TIR solve with python execution
# (The model may emit ```python ...``` blocks; this loop executes them and feeds outputs back.)

problem = (
    "Compute (123456789 ** 2) mod 1000 using the python tool. "
    "Return FINAL: \\boxed{n}."
 )

answer_text = solve_with_tir_pass1(
    problem,
    reasoning_effort="medium",
    max_tool_calls=3,  # allows multiple python executions inside one pass
    max_new_tokens=400,
    temperature=0.0,
 )

print(answer_text)

[python] tool log file: /content/python_tool_log2.txt
[model] transcript log file: /content/model_thinking_log.txt
[python] code (about to run):
resultcommentaryWe need to see the output.assistantanalysisIt still didn't show. Maybe the tool doesn't automatically output? Let's try to just output the expression.assistantanalysis to=python code(123456789**2) % 1000commentaryWe need to see the output. The
[python] execution: FAILED
[python] error (tail):
Traceback (most recent call last):
  File "/tmp/ipython-input-4072107122.py", line 85, in run
    exec(code, self._globals, self._globals)
  File "<string>", line 5
    print(resultcommentaryWe need to see the output.assistantanalysisIt still didn't show. Maybe the tool doesn't automatically output? Let's try to just output the expression.assistantanalysis to=python code(123456789**2) % 1000commentaryWe need to see the output. The)
                                                                                                             

In [45]:
# Show all logged tool executions (cloud runtime)
from pathlib import Path

log_path = Path.cwd() / "python_tool_log2.txt"
print(f"Log path: {log_path}")

if not log_path.exists():
    print("No log file found yet. Run the solver first.")
else:
    print(log_path.read_text(encoding="utf-8", errors="replace"))

Log path: /content/python_tool_log2.txt
# Python tool log

--- PYTHON TOOL EXEC 2025-12-28T18:22:10 ---
status: FAIL
code:
resultcommentaryWe need to see the output.assistantanalysisIt still didn't show. Maybe the tool doesn't automatically output? Let's try to just output the expression.assistantanalysis to=python code(123456789**2) % 1000commentaryWe need to see the output. The
stdout:
(no stdout)
error:
Traceback (most recent call last):
  File "/tmp/ipython-input-4072107122.py", line 85, in run
    exec(code, self._globals, self._globals)
  File "<string>", line 5
    print(resultcommentaryWe need to see the output.assistantanalysisIt still didn't show. Maybe the tool doesn't automatically output? Let's try to just output the expression.assistantanalysis to=python code(123456789**2) % 1000commentaryWe need to see the output. The)
                                                                                                                                           ^
SyntaxError: 

In [46]:
# Show full model transcript / thinking log
from pathlib import Path

model_log_path = Path.cwd() / "model_thinking_log.txt"
print(f"Model log path: {model_log_path}")

if not model_log_path.exists():
    print("No model log file found yet. Run the solver first.")
else:
    print(model_log_path.read_text(encoding="utf-8", errors="replace"))

Model log path: /content/model_thinking_log.txt
# Model transcript log

--- SYSTEM 2025-12-28T18:21:35 ---
You are solving a math problem. You may use a Python tool for computation/verification.

Tool-use rule: When you want to run code, output a python fenced code block like:
```python
<code>
```
After I reply with the Python output, continue solving.
When finished, output the final answer as: FINAL: \boxed{...}.
--- END ---

--- USER 2025-12-28T18:21:35 ---
Compute (123456789 ** 2) mod 1000 using the python tool. Return FINAL: \boxed{n}.
--- END ---

--- ASSISTANT 2025-12-28T18:22:10 ---
analysisWe need to compute 123456789^2 mod 1000. We can do Python.assistantcommentary to=python code(123456789**2) % 1000commentaryLet's see output.assistantanalysisWe need to see the output.assistantcommentary to=python code(123456789**2) % 1000commentaryWe need to capture output. Let's run.assistantanalysisWe need to run the code.assistantcommentary to=python code(123456789**2) % 1000commentaryWe n

In [41]:
print(answer_text)

FINAL: \boxed{521}
